# Reinforcement Learning Based Trading Agent

Follow the instructions step by step and fill in the TODOs


## 1. Install and Import Libraries

In [1]:

# Uncomment only if needed
# !pip install yfinance numpy pandas matplotlib

import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt


## 2. Download Market Data

In [4]:

# TODO: choose a stock symbol (e.g., "AAPL", "MSFT", "GOOG")
symbol = "AAPL"

# TODO: download historical stock price data using yfinance
# Hint: use yf.download with a start and end date
data = yf.download(symbol, start="2020-01-01", end="2021-01-01")

# TODO: extract ONLY the closing prices
# IMPORTANT: flatten the array so each price is a scalar (fixes NumPy state error)
prices = data["Close"].values.flatten()

print("Trading days:", len(prices))
print("Sample price:", prices[0], type(prices[0]))


/tmp/ipython-input-3445520733.py:6: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(symbol, start="2020-01-01", end="2021-01-01")
[*********************100%***********************]  1 of 1 completed

Trading days: 253
Sample price: 72.46826934814453 <class 'numpy.float64'>


## 3. Trading Environment

In [7]:
# TODO: Define a custom trading environment for Reinforcement Learning

class TradingEnv:
    def __init__(self, prices):
        # TODO: store historical prices
        self.prices = prices

        # TODO: reset environment to initial state
        self.reset()

    def reset(self):
        # TODO: initialize time step
        self.t = 0
        # TODO: initialize starting cash
        self.cash = 10000

        # TODO: initialize stock holding
        # 0 = no stock, 1 = holding stock
        self.stock = 0

        # TODO: initialize done flag
        self.done = False

        # TODO: return initial state
        return self._get_state()

    def _get_state(self):
        # TODO: define state as a NumPy array
        # State should contain:
        # 1. current price
        # 2. stock holding (0 or 1)
        return np.array([self.prices[self.t], self.stock])

    def step(self, action):
        # TODO: get current stock price
        price = self.prices[self.t]

        # TODO: define action logic
        # Action 0 → Hold (do nothing)
        # Action 1 → Buy (only if enough cash)
        # Action 2 → Sell (only if holding stock)
        if action==1 and self.cash>=price:
            self.cash -= price
            self.stock = 1
        elif action==2 and self.stock==1:
            self.cash += price
            self.stock = 0

        # TODO: move to next time step
        self.t+=1

        # TODO: check termination condition
        if self.t>=len(self.prices)-1:
            self.done = True


        # TODO: define reward (portfolio value)
        reward = self.cash + self.stock*self.prices[self.t]

        # TODO: get next state
        next_state = self._get_state()
        # TODO: return next_state, reward, done
        return next_state,reward, self.done


## 4. Q-Learning Setup

In [14]:

# TODO: Initialize the Q-table
# Hint: number of states = number of time steps
# Hint: number of actions = 3 (Hold, Buy, Sell)
Q = np.zeros((len(prices),3))
# TODO: set learning rate (alpha)
alpha = 0.1

# TODO: set discount factor (gamma)
gamma = 0.95
# TODO: set exploration rate (epsilon)
epsilon = 0.1


## 5. Train the Agent

In [16]:

# TODO: create trading environment
env = TradingEnv(prices)

# TODO: set number of training episodes
episodes = 100

# TODO: training loop
for episode in range(episodes):

    # TODO: reset environment at start of each episode
    env.reset()

    # TODO: loop until episode ends
    while not env.done:

        # TODO: get current state index (time step)
        t = env.t

        # TODO: epsilon-greedy action selection
        if np.random.rand()<epsilon:
            action = np.random.choice([0,1,2])
        else:
            action = np.argmax(Q[t])

        # TODO: take action in environment
        _, reward, _ = env.step(action)

        # TODO: update Q-value using Bellman equation
        Q[t, action] += alpha * (reward + gamma * np.max(Q[min(t+1, len(prices)-1)]) - Q[t,action])

# TODO: indicate training completion
print("Training complete!")


Training complete!


## 6. Evaluate Trained Agent

In [17]:

# TODO: create a new environment for evaluation
env =TradingEnv(prices)

# TODO: run the trained agent without exploration
while not env.done:

    # TODO: get current state index (time step)
    t = env.t

    # TODO: select best action from Q-table
    action = np.argmax(Q[t])

    # TODO: apply action in environment
    env.step(action)

# TODO: compute final portfolio value
final_value =env.cash + env.stock*prices[-1]

# TODO: print final result
print("Final RL Agent Portfolio Value:", final_value)



Final RL Agent Portfolio Value: 9709.952507019043


## 7. Buy and Hold Baseline

In [18]:
# TODO: implement Buy-and-Hold baseline strategy
# Instructions:
# - Buy one stock on the first day
# - Hold it until the last day
# - Start with initial cash of 10000
initial_cash = 10000
shares = initial_cash//prices[0]
remaining_cash = initial_cash-shares*prices[0]

buy_and_hold_value = shares*prices[-1] + remaining_cash

# TODO: print Buy-and-Hold portfolio value
print("Buy and Hold Portfolio Value:", buy_and_hold_value)



Buy and Hold Portfolio Value: 17767.773109436035
